# Connecting neo4j and Databricks

In [0]:
%pip install neo4j==6.3.0

In [0]:
dbutils.library.restartPython()

In [0]:
from neo4j import GraphDatabase

NEO4J_URI = dbutils.secrets.get(
    "healthcare-graphrag",
    "neo4j-uri"
).strip()

NEO4J_USERNAME = dbutils.secrets.get(
    "healthcare-graphrag",
    "neo4j-username"
).strip()

NEO4J_PASSWORD = dbutils.secrets.get(
    "healthcare-graphrag",
    "neo4j-password"
)

NEO4J_DATABASE = "neo4j"

assert NEO4J_URI.startswith("neo4j+s://")
assert "127.0.0.1" not in NEO4J_URI
assert "localhost" not in NEO4J_URI

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully to Neo4j AuraDB")

In [0]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully to Neo4j AuraDB")

# Configuration and imports

In [0]:
from decimal import Decimal
from datetime import date, datetime
import math

CATALOG = "patient_kg_dev"
GRAPH = f"{CATALOG}.graph_ready"
NEO4J_DATABASE = "neo4j"
BATCH_SIZE = 200

node_tables = {
    "Patient": f"{GRAPH}.patient_nodes",
    "Encounter": f"{GRAPH}.encounter_nodes",
    "ConditionEvent": f"{GRAPH}.condition_nodes",
    "MedicationEvent": f"{GRAPH}.medication_nodes",
    "ProcedureEvent": f"{GRAPH}.procedure_nodes"
}

In [0]:
driver.execute_query(
    """
    CREATE CONSTRAINT entity_node_id_unique IF NOT EXISTS
    FOR (n:Entity)
    REQUIRE n.node_id IS UNIQUE
    """,
    database_=NEO4J_DATABASE
)

print("Neo4j node constraint is ready")

In [0]:
def clean_neo4j_value(value):
    if value is None:
        return None

    if isinstance(value, Decimal):
        return float(value)

    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None

    if isinstance(value, (str, int, bool, float, date, datetime)):
        return value

    return str(value)


def spark_nodes_to_parameters(dataframe):
    parameter_rows = []

    for spark_row in dataframe.collect():
        row = spark_row.asDict(recursive=True)

        node_id = str(row.pop("node_id"))
        row.pop("node_label", None)

        properties = {
            key: clean_neo4j_value(value)
            for key, value in row.items()
            if value is not None
        }

        parameter_rows.append({
            "node_id": node_id,
            "properties": properties
        })

    return parameter_rows


def chunks(rows, batch_size):
    for start in range(0, len(rows), batch_size):
        yield rows[start:start + batch_size]

In [0]:
ALLOWED_NODE_LABELS = {
    "Patient",
    "Encounter",
    "ConditionEvent",
    "MedicationEvent",
    "ProcedureEvent"
}


def load_node_table(label, table_name):
    if label not in ALLOWED_NODE_LABELS:
        raise ValueError(f"Unsupported Neo4j label: {label}")

    dataframe = spark.table(table_name)
    rows = spark_nodes_to_parameters(dataframe)

    query = f"""
    UNWIND $rows AS row

    MERGE (n:Entity:{label} {{
        node_id: row.node_id
    }})

    SET n += row.properties
    """

    processed = 0

    for batch in chunks(rows, BATCH_SIZE):
        driver.execute_query(
            query,
            parameters_={"rows": batch},
            database_=NEO4J_DATABASE
        )

        processed += len(batch)

    print(
        f"{label}: loaded {processed} nodes "
        f"from {table_name}"
    )

    return processed

In [0]:
loaded_node_counts = {}

for label, table_name in node_tables.items():
    loaded_node_counts[label] = load_node_table(
        label,
        table_name
    )

print("Node loading completed")
print(loaded_node_counts)

In [0]:
node_records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    RETURN
        labels(n) AS labels,
        count(*) AS node_count
    ORDER BY node_count DESC
    """,
    database_=NEO4J_DATABASE
)

for record in node_records:
    print(record["labels"], record["node_count"])

In [0]:
total_node_records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    RETURN count(n) AS total_nodes
    """,
    database_=NEO4J_DATABASE
)

neo4j_node_count = total_node_records[0]["total_nodes"]

print("Total Neo4j nodes:", neo4j_node_count)

assert neo4j_node_count == 671, (
    f"Expected 671 nodes, found {neo4j_node_count}"
)

# Prepare Relationships

In [0]:
relationship_df = spark.table(
    f"{GRAPH}.relationships"
)

display(
    relationship_df
    .groupBy("relationship_type")
    .count()
    .orderBy("relationship_type")
)

In [0]:
def spark_relationships_to_parameters(dataframe):
    grouped_relationships = {}

    for spark_row in dataframe.collect():
        row = spark_row.asDict(recursive=True)

        relationship_id = str(row.pop("relationship_id"))
        source_node_id = str(row.pop("source_node_id"))
        target_node_id = str(row.pop("target_node_id"))
        relationship_type = str(row.pop("relationship_type"))

        properties = {
            key: clean_neo4j_value(value)
            for key, value in row.items()
            if value is not None
        }

        parameter = {
            "relationship_id": relationship_id,
            "source_node_id": source_node_id,
            "target_node_id": target_node_id,
            "properties": properties
        }

        grouped_relationships.setdefault(
            relationship_type,
            []
        ).append(parameter)

    return grouped_relationships

# Relationship-loading function

In [0]:
ALLOWED_RELATIONSHIP_TYPES = {
    "HAS_ENCOUNTER",
    "HAS_CONDITION",
    "HAS_MEDICATION",
    "HAS_PROCEDURE"
}


def load_relationship_type(relationship_type, rows):
    if relationship_type not in ALLOWED_RELATIONSHIP_TYPES:
        raise ValueError(
            f"Unsupported relationship type: {relationship_type}"
        )

    query = f"""
    UNWIND $rows AS row

    MATCH (source:Entity {{
        node_id: row.source_node_id
    }})

    MATCH (target:Entity {{
        node_id: row.target_node_id
    }})

    MERGE (source)-[relationship:{relationship_type} {{
        relationship_id: row.relationship_id
    }}]->(target)

    SET relationship += row.properties
    """

    processed = 0

    for batch in chunks(rows, BATCH_SIZE):
        driver.execute_query(
            query,
            parameters_={"rows": batch},
            database_=NEO4J_DATABASE
        )

        processed += len(batch)

    print(
        f"{relationship_type}: loaded "
        f"{processed} relationships"
    )

    return processed

# Load all relationships

In [0]:
relationships_by_type = (
    spark_relationships_to_parameters(
        relationship_df
    )
)

loaded_relationship_counts = {}

for relationship_type, rows in relationships_by_type.items():
    loaded_relationship_counts[relationship_type] = (
        load_relationship_type(
            relationship_type,
            rows
        )
    )

print("Relationship loading completed")
print(loaded_relationship_counts)

# Realationship counts

In [0]:
relationship_records, _, _ = driver.execute_query(
    """
    MATCH ()-[r]->()
    RETURN
        type(r) AS relationship_type,
        count(*) AS relationship_count
    ORDER BY relationship_type
    """,
    database_=NEO4J_DATABASE
)

for record in relationship_records:
    print(
        record["relationship_type"],
        record["relationship_count"]
    )

In [0]:
total_relationship_records, _, _ = driver.execute_query(
    """
    MATCH ()-[r]->()
    RETURN count(r) AS total_relationships
    """,
    database_=NEO4J_DATABASE
)

neo4j_relationship_count = (
    total_relationship_records[0]["total_relationships"]
)

print(
    "Total Neo4j relationships:",
    neo4j_relationship_count
)

assert neo4j_relationship_count == 1079, (
    f"Expected 1079 relationships, "
    f"found {neo4j_relationship_count}"
)

# Verify Label Counts

In [0]:
expected_label_counts = {
    "Patient": 13,
    "Encounter": 62,
    "ConditionEvent": 194,
    "MedicationEvent": 143,
    "ProcedureEvent": 259
}

for label, expected_count in expected_label_counts.items():
    records, _, _ = driver.execute_query(
        f"""
        MATCH (n:Entity:{label})
        RETURN count(n) AS node_count
        """,
        database_=NEO4J_DATABASE
    )

    actual_count = records[0]["node_count"]

    print(
        label,
        "| expected:", expected_count,
        "| actual:", actual_count
    )

    assert actual_count == expected_count

# Verify relationship endpoints

In [0]:
missing_endpoint_records, _, _ = driver.execute_query(
    """
    MATCH (source:Entity)-[r]->(target:Entity)
    WHERE source.node_id IS NULL
       OR target.node_id IS NULL
    RETURN count(r) AS invalid_relationships
    """,
    database_=NEO4J_DATABASE
)

invalid_relationship_count = (
    missing_endpoint_records[0]["invalid_relationships"]
)

print(
    "Invalid relationship endpoints:",
    invalid_relationship_count
)

assert invalid_relationship_count == 0

# Inspect a patient-centered graph

In [0]:
sample_records, _, _ = driver.execute_query(
    """
    MATCH (patient:Patient)-[relationship]->(event:Entity)
    RETURN
        patient.patient_id AS patient_id,
        type(relationship) AS relationship_type,
        labels(event) AS event_labels,
        event.description_source AS description,
        coalesce(
            event.start_at,
            event.start_date
        ) AS event_start
    ORDER BY event_start
    LIMIT 25
    """,
    database_=NEO4J_DATABASE
)

for record in sample_records:
    print(record.data())

In [0]:
ihd_records, _, _ = driver.execute_query(
    """
    MATCH (patient:Patient)
          -[:HAS_CONDITION]->
          (ihd:ConditionEvent)

    WHERE ihd.code = $ihd_code

    OPTIONAL MATCH (patient)-[relationship]->(event:Entity)

    WHERE event.node_id <> ihd.node_id

    RETURN
        patient.patient_id AS patient_id,
        ihd.start_date AS ihd_diagnosis_date,
        type(relationship) AS relationship_type,
        labels(event) AS event_labels,
        event.description_source AS event_description,
        coalesce(
            event.start_at,
            event.start_date
        ) AS event_start

    ORDER BY patient_id, event_start
    LIMIT 50
    """,
    ihd_code="414545008",
    database_=NEO4J_DATABASE
)

for record in ihd_records:
    print(record.data())

In [0]:
print("NEO4J LOAD COMPLETED SUCCESSFULLY")
print("Nodes:", neo4j_node_count)
print("Relationships:", neo4j_relationship_count)
print("Database:", NEO4J_DATABASE)